In [58]:
"""
HomeGuard Security System Simulator
Author: [Mira Raab]
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""
 
import random
from datetime import datetime
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

In [59]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    sensor = {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold
    }
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    return alert

# Initialize sensors for the Peterson home
sensors = [
    create_sensor("S001", "Living Room", "motion"),
    create_sensor("S002", "Kitchen", "temperature", threshold=95),
    create_sensor("S003", "Front Door", "door"),
    create_sensor("S004", "Bedroom", "smoke")
]
print(sensors)

[{'id': 'S001', 'location': 'Living Room', 'type': 'motion', 'threshold': None}, {'id': 'S002', 'location': 'Kitchen', 'type': 'temperature', 'threshold': 95}, {'id': 'S003', 'location': 'Front Door', 'type': 'door', 'threshold': None}, {'id': 'S004', 'location': 'Bedroom', 'type': 'smoke', 'threshold': None}]


In [60]:
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['id']}: {sensor['location']} ({sensor['type']})")

Initialized 4 sensors
  - S001: Living Room (motion)
  - S002: Kitchen (temperature)
  - S003: Front Door (door)
  - S004: Bedroom (smoke)


In [61]:
def is_abnormal_reading(sensor, reading_value):
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    elif sensor_type == "motion":
        return reading_value == True
    elif sensor_type == "door":
        return reading_value == "OPEN"
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"
    else:
        return False

In [62]:
def should_trigger_security_alert(sensor, reading_value, system_mode):
    if is_abnormal_reading(sensor, reading_value):
        if system_mode == "AWAY":
            return True
        elif system_mode == "HOME":
            # Example: Only trigger if it's a smoke sensor
            return sensor["type"] == "smoke"
        elif system_mode == "SLEEP":
            # Example: Trigger for motion or smoke sensors
            return sensor["type"] in ["motion", "smoke"]
        
    return False

In [63]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False


In [64]:
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [65]:
import random

def generate_reading(sensor):
    """
    Generates a realistic reading for a sensor.

    Parameters:
    - sensor: Sensor dictionary

    Returns:
    - A realistic reading value based on sensor type
    """
    sensor_type = sensor["type"]

    # Temperature: random float between 30–100°F
    if sensor_type == "Temperature":
        return round(random.uniform(30, 100), 1)

    # Motion: random boolean
    elif sensor_type == "Motion":
        return random.choice([True, False])

    # Door: random choice between "OPEN" and "CLOSED"
    elif sensor_type == "Door":
        return random.choice(["OPEN", "CLOSED"])

    # Smoke: random choice between "CLEAR" and "DETECTED"
    elif sensor_type == "Smoke":
        return random.choice(["CLEAR", "DETECTED"])

In [66]:
def process_reading(sensor, reading_value, system_mode):
    """
    Processes a sensor reading and determines if alerts are needed.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The reading from the sensor
    - system_mode: Current system mode
    
    Returns:
    - A list of alerts (empty if no alerts needed)
    """
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")

    sensor_type = sensor["type"]

    # Security alerts
    if system_mode == "AWAY":
        if sensor_type == "motion" and reading_value == True:
            alerts.append({
                "severity": "HIGH",
                "message": f"Motion detected at {timestamp}!",
            })
        if sensor_type == "door" and reading_value == "OPEN":
            alerts.append({
                "severity": "HIGH",
                "message": f"Door opened at {timestamp}!",
            })

    # Safety alerts
    if sensor_type == "smoke" and reading_value == "DETECTED":
        alerts.append({
            "severity": "CRITICAL",
            "message": "Smoke detected! Take action immediately!",
        })
    if sensor_type == "temperature" and (reading_value < 18 or reading_value > 40):
        alerts.append({
            "severity": "MEDIUM",
            "message": f"Extreme temperature: {reading_value}°C",
        })

    # Comfort notifications (only in HOME mode)
    if system_mode == "HOME":
        if sensor_type == "temperature" and (reading_value < 20 or reading_value > 25):
            alerts.append({
                "severity": "LOW",
                "message": f"Temperature out of comfort range: {reading_value}°C",
            })

    return alerts

In [67]:


def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }

    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")


def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")


In [68]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: None
[ALERT!] 🚨 HIGH: Motion detected at 15:51:05!


In [69]:
import random

class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """

    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.sensor_type = sensor_type
        self.threshold = threshold
        self.current_value = None

    def read(self):
        if self.sensor_type == "temperature":
            self.current_value = random.randint(15, 45)
        elif self.sensor_type == "motion":
            self.current_value = random.choice([True, False])
        elif self.sensor_type == "door":
            self.current_value = random.choice(["open", "closed"])
        elif self.sensor_type == "smoke":
            self.current_value = random.choice([True, False])
        return self.current_value

    def isAbnormal(self):
        if self.sensor_type == "temperature":
            return self.current_value > self.threshold
        elif self.sensor_type == "motion":
            return self.current_value == True
        elif self.sensor_type == "door":
            return self.current_value == "open"
        elif self.sensor_type == "smoke":
            return self.current_value == True

    def reset(self):
        self.current_value = None

    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"


sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "door"),
    Sensor("SMOKE_001", "Bedroom", "smoke")
]

In [70]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 20
Is abnormal: False
Sensor info: TEST_001 (Test Room): 20


In [71]:
import time
from datetime import datetime
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.

    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")

    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]

    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        for sensor in sensors:
            reading = sensor.read()

            if sensor.sensor_type == "temperature":
                status = "Normal" if reading <= sensor.threshold else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°C ({status})")
            elif sensor.sensor_type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.sensor_type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.sensor_type == "smoke":
                status = "DETECTED" if reading else "Clear"
                print(f"[READING] {sensor.location} Smoke: {status}")

            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.sensor_type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)

            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

        time.sleep(0.5)

    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)


if __name__ == "__main__":
    run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 15:51:05
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Motion detected at 15:51:05!
[LOG] [15:51:05] Sending notification to homeowner...
[READING] Kitchen Temperature: 31°C (Normal)
[READING] Front Door: open
[READING] Bedroom Smoke: DETECTED

Time: 15:51:06
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 33°C (Normal)
[READING] Front Door: open
[READING] Bedroom Smoke: Clear

Time: 15:51:06
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 25°C (Normal)
[READING] Front Door: open
[READING] Bedroom Smoke: Clear

Simulation complete!
